# Day 5 — Delta: History, Time Travel, Optimize & Ops

Allow about 1.5–2 hours. Sections on CDF (6b), CHECK (7b), and the recap quiz can be done later if you run short on time.

Use notebook 01 first so `P_BASIC` and `P_PART` exist.

You will inspect history, use version and timestamp time travel, call `DeltaTable.history`, read about RESTORE and shallow clone (commented patterns), run OPTIMIZE and optional ZORDER, review VACUUM and CDF concepts, try optional hands-on cells for CDF and CHECK where your runtime allows, and work through extra drills and a short quiz.

Standard Delta versioning and optimization labs from Databricks training map well to this content.

## Setup

In [0]:
BASE_PATH = "abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/"
STUDENT_ID = "u01"  # Change to your assigned u01–u16
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
P_PART = DAY5 + "/delta/flight_partitioned"

# Verify prerequisite tables from Notebook 01 exist
print("=== Checking Day 5 Notebook 01 Prerequisites ===")
prereqs_met = True
try:
    spark.read.format("delta").load(P_BASIC).limit(1).collect()
    print(f"✓ P_BASIC table exists: {P_BASIC}")
except Exception as e:
    print(f"✗ P_BASIC table NOT found: {P_BASIC}")
    print(f"  Error: {type(e).__name__}")
    print(f"  Action: Run notebook 01 first to create this table")
    prereqs_met = False

try:
    spark.read.format("delta").load(P_PART).limit(1).collect()
    print(f"✓ P_PART table exists: {P_PART}")
except Exception as e:
    print(f"✗ P_PART table NOT found: {P_PART}")
    print(f"  Error: {type(e).__name__}")
    print(f"  Action: Run notebook 01 first to create this table")
    prereqs_met = False

if not prereqs_met:
    print("\n⚠️  WARNING: Day 5 Notebook 01 prerequisite tables are missing.")
    print("   Complete notebook 01 before running this notebook.")
else:
    print("\n✓ All prerequisites met for Day 5 Notebook 03")

print("\nUsing:", P_BASIC)

## Section 1 — Deep **history** inspection

In [0]:
spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`").display(50, truncate=False)

In [0]:
# Latest version number
from pyspark.sql.functions import col as C

h = spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`")
ver_max = h.selectExpr("max(version) as v").collect()[0]["v"]
ver_min = h.selectExpr("min(version) as v").collect()[0]["v"]
print("version min/max:", ver_min, ver_max)

## Section 2 — **Time travel** reads

In [0]:
from pyspark.sql.functions import col

v = int(spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`").selectExpr("max(version) as v").collect()[0]["v"])
if v >= 1:
    print("sum(count) at version v-1:")
    spark.read.format("delta").option("versionAsOf", v - 1).load(P_BASIC).groupBy().sum("count").display()
print("sum(count) at latest version", v)
spark.read.format("delta").option("versionAsOf", v).load(P_BASIC).groupBy().sum("count").display()

In [0]:
from pyspark.sql.functions import col as C

row = spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`").filter(C("version") == 0).select("timestamp").collect()
if row:
    ts = str(row[0]["timestamp"])
    c_ts = spark.read.format("delta").option("timestampAsOf", ts).load(P_BASIC).count()
    print("Row count TIMESTAMP AS OF version-0 time:", c_ts)
else:
    print("No history row for version 0")

## Section 2b — **Subtract** rows: latest vs previous version

In [0]:
from pyspark.sql.functions import col

h = spark.sql(f"DESCRIBE HISTORY delta.`{P_BASIC}`")
v = int(h.selectExpr("max(version) as v").collect()[0]["v"])
cols = ["DEST_COUNTRY_NAME", "ORIGIN_COUNTRY_NAME", "count"]
cur = spark.read.format("delta").option("versionAsOf", v).load(P_BASIC).select(*cols)
if v >= 1:
    prev = spark.read.format("delta").option("versionAsOf", v - 1).load(P_BASIC).select(*cols)
    print("counts prev, cur:", prev.count(), cur.count())
    print("in cur but not prev:", cur.subtract(prev).count())
    print("in prev but not cur:", prev.subtract(cur).count())
else:
    print("Only one version; run notebook 01 append cells first.")

## Section 2c — **`DeltaTable` API** for history (same log as SQL)

In [0]:
from delta.tables import DeltaTable

DeltaTable.forPath(spark, P_BASIC).history(15).select("version", "timestamp", "operation").display(15, truncate=False)

## Section 3 — **OPTIMIZE** + **ZORDER** (Databricks)

In [0]:
# File metrics before OPTIMIZE
spark.sql(f"DESCRIBE DETAIL delta.`{P_PART}`").select("numFiles", "sizeInBytes").display(truncate=False)

In [0]:
# Compacts small files (Databricks Delta)
spark.sql(f"OPTIMIZE delta.`{P_PART}`")

In [0]:
# Z-order by high-cardinality filter columns (example)
try:
    spark.sql(f"OPTIMIZE delta.`{P_PART}` ZORDER BY (DEST_COUNTRY_NAME)")
except Exception as e:
    print("ZORDER may require Photon/Delta version:", type(e).__name__, e)

In [0]:
spark.sql(f"DESCRIBE DETAIL delta.`{P_PART}`").select("numFiles", "sizeInBytes").display(truncate=False)

## Section 4b — **OPTIMIZE** on **`P_BASIC`** (compare detail)

In [0]:
spark.sql(f"DESCRIBE DETAIL delta.`{P_BASIC}`").select("numFiles", "sizeInBytes").display(truncate=False)
spark.sql(f"OPTIMIZE delta.`{P_BASIC}`")
spark.sql(f"DESCRIBE DETAIL delta.`{P_BASIC}`").select("numFiles", "sizeInBytes").display(truncate=False)

## Section 5 — **VACUUM** (discussion only)

`VACUUM` deletes data files **no longer referenced** by the table. Default **7-day** retention for time travel.

```python
# spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
# spark.sql(f"VACUUM delta.`{P_BASIC}` RETAIN 168 HOURS")
```
